# 📖 Notebook 3: Services & Networking — How Traffic Reaches Your Pods

In Notebook 02, you learned how Deployments keep pods running. In this notebook, you will learn how those pods talk to each other and how traffic enters the cluster. Think of this notebook as the road system for your microservices: Services are the stable roads inside the cluster, and Ingress is the front door from the outside world.

> **Prerequisite: Notebook 02 must be finished.** It creates the `k8s-lab` namespace and
> the `api-gateway`, `user-service` and `order-service` deployments that everything below
> assumes. If `kubectl get deploy -n k8s-lab` comes back empty, go back and run
> Notebook 02's cleanup cell — that is the cell that applies the shared manifests.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['minikube', 'kubectl']
INSTALL_HINTS = {
    'minikube': 'https://minikube.sigs.k8s.io/docs/start/  (or `brew install minikube`)',
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers used throughout this notebook ────────────────────────────────
# `!kubectl ...` prints, but a non-zero exit does not fail the cell. Anything
# this notebook *claims* is therefore also checked in Python.
import contextlib
import json
import socket
import subprocess
import time
import urllib.error
import urllib.request

NS = "k8s-lab"


def kget(*args, ns=NS):
    cmd = ["kubectl", "get", *args, "-o", "json"] + (["-n", ns] if ns else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def endpoint_ips(name, ns=NS):
    """The addresses actually behind a Service. Empty means nothing is served."""
    r = subprocess.run(["kubectl", "get", "endpoints", name, "-n", ns, "-o", "json"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        return []
    return [a["ip"]
            for s in json.loads(r.stdout).get("subsets", []) or []
            for a in s.get("addresses", []) or []]


def wait_until(predicate, timeout=170, interval=5, what="condition"):
    """Poll until predicate() is truthy. Returns its value; raises on timeout.

    Notebook runners cap how long a single cell may block (180s is common), so
    long waits are written as a bounded poll that can be continued in the next
    cell rather than one `kubectl wait --timeout=300s` that blows the budget."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    return None


def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


@contextlib.contextmanager
def port_forward(target, remote_port, ns=NS):
    """`kubectl port-forward` as a context manager.

    This is the portable way to reach an in-cluster Service from your laptop. It
    works identically on minikube, kind, Docker Desktop and a real cloud cluster,
    which is more than can be said for a NodePort (see the next section)."""
    local = free_port()
    proc = subprocess.Popen(
        ["kubectl", "port-forward", target, f"{local}:{remote_port}", "-n", ns],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        deadline = time.time() + 30
        while time.time() < deadline:
            with socket.socket() as s:
                s.settimeout(0.5)
                if s.connect_ex(("127.0.0.1", local)) == 0:
                    break
            time.sleep(0.5)
        else:
            raise RuntimeError(f"port-forward to {target} never came up")
        yield f"http://127.0.0.1:{local}"
    finally:
        proc.terminate()
        proc.wait(timeout=10)


def http_get(url, headers=None, timeout=10, retries=10):
    """GET with retries -- an Ingress controller needs a few seconds to notice a
    new Ingress object, and a fresh port-forward needs one to settle."""
    last = None
    for _ in range(retries):
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, r.read().decode()
        except urllib.error.HTTPError as e:
            last = (e.code, e.read().decode()[:300])
        except Exception as e:  # noqa: BLE001 - connection refused, DNS, timeout
            last = (None, repr(e))
        time.sleep(3)
    return last


print("helpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why Kubernetes Services exist and why pod IP addresses are not enough
- Create and test a **ClusterIP** Service for internal communication
- Create and test a **NodePort** Service for local external access in minikube
- Understand Kubernetes DNS names such as `user-service.k8s-lab.svc.cluster.local`
- Apply the shared Service manifests for `api-gateway`, `user-service`, and `order-service`
- Test communication between microservices inside the `k8s-lab` namespace
- Enable the NGINX Ingress Controller in minikube and route HTTP traffic through an Ingress resource
- Explain why a **LoadBalancer** Service is the right answer in a cloud and a wrong one in minikube
- Use a **headless** Service (`clusterIP: None`) and say how it differs from ClusterIP
- Debug the classic silent failure: a Service whose selector matches no pods
- Clean up the networking resources you created

## 🛠️ Setup

Before starting, make sure your local lab environment is ready.

1. Open this notebook from the `kubernetes/notebooks/` folder — every relative path below
   (`../manifests/...`) is written from there.
2. Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it
   doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.
3. Confirm that minikube is running and that the apps from Notebook 02 are already
   deployed in the `k8s-lab` namespace.
4. If Notebook 02 is not finished yet, go back and deploy the three sample FastAPI apps
   first.

The commands below check the cluster, namespace, deployments, and pods.

In [ ]:
!kubectl config current-context
!kubectl get namespace k8s-lab
!kubectl get deploy,pods -n k8s-lab -o wide

## Why Pod IPs Are Not Enough
A pod is **ephemeral**, which means Kubernetes is free to replace it at any time. If a node fails or you roll out a new version, the old pod can disappear and a new pod will be created with a different IP address.

That creates a problem: if `api-gateway` tries to call `user-service` by a pod IP, that connection can break as soon as Kubernetes replaces the pod.

A **Service** solves this by giving a workload a **stable name** and a **stable virtual IP**. The pods behind the Service can come and go, but the Service name stays the same. Your applications talk to the Service, and Kubernetes forwards traffic to healthy pods behind it.

A good mental model is this:

- **Pods** are the workers
- **Services** are the stable phone numbers for those workers
- **Ingress** is the public receptionist at the front desk

In [ ]:
!kubectl get pods -n k8s-lab -o wide

# Exercise: look at the pod IPs above.
# If one of these pods is recreated, its IP can change.
# That is exactly why we use a Service instead of hard-coding pod addresses.

## One Service, Many Pods
Here is the basic idea behind a Kubernetes Service. One stable Service sits in front of multiple pods. Kubernetes load balances requests across the healthy pods for you.

```text
+-----------+        +-------------------+        +-----------------+
| External  | -----> | Service           | -----> | Pod 1           |
| Request   |        | user-service      | -----> | Pod 2           |
| or Client |        | Stable virtual IP | -----> | Pod 3           |
+-----------+        +-------------------+        +-----------------+
```

The client does **not** need to know which exact pod handled the request. It only needs to know the Service name.

In [ ]:
!kubectl get deploy user-service -n k8s-lab
!kubectl get pods -l app=user-service -n k8s-lab -o wide

# Exercise: count how many user-service pods exist.
# A Service can sit in front of one pod or many pods.

## ClusterIP — the Default, for Traffic Inside the Cluster
`ClusterIP` is the default Service type in Kubernetes. It creates an internal virtual IP that other pods can reach, but it does **not** expose the app outside the cluster.

This is perfect for internal microservice-to-microservice communication. In our lab, `user-service` should be reachable from other workloads inside the cluster, but it does not need to be public.

In the next cell, you will create a `ClusterIP` Service named `user-service` and then inspect it.

In [ ]:
!kubectl expose deployment user-service --name user-service --type=ClusterIP --port=8001 --target-port=8001 -n k8s-lab --dry-run=client -o yaml | kubectl apply -f -
!kubectl get svc user-service -n k8s-lab
!kubectl describe svc user-service -n k8s-lab

## Test It From Inside the Cluster
A Service is only useful if other workloads can reach it. The easiest way to test this is to launch a temporary helper pod that already contains `curl`.

The command below starts a short-lived pod, sends an HTTP request to `user-service`, prints the response, and then removes the helper pod automatically.

In [ ]:
# `--rm` deletes the pod when the command finishes, but kubectl refuses `--rm`
# unless the client is attached to the container: without `-i` you get
#   error: --rm should only be used for attached containers
# So `-i` is not optional here.
out = subprocess.run(
    ["kubectl", "run", "curl-demo", "-i", "--rm", "--restart=Never",
     "--image=curlimages/curl:8.7.1", "-n", NS, "--command", "--",
     "curl", "-s", "http://user-service:8001/health"],
    capture_output=True, text=True, timeout=180)
print(out.stdout)

assert '"status":"ok"' in out.stdout, (
    "user-service did not answer through its ClusterIP.\n"
    f"stdout: {out.stdout[:400]}\nstderr: {out.stderr[:400]}"
)
print("✅ one pod reached another purely by Service name")

## NodePort — Reaching the Cluster From Your Laptop
A `NodePort` Service opens the same port on **every node** in the cluster, in the range
30000–32767, and forwards it to the Service. It is a superset of ClusterIP: the Service
still has its internal virtual IP as well.

For this lab, we expose `api-gateway` with a `NodePort` Service. That gives us a stable
entry point for local testing before we move on to Ingress.

> **"Every node" means the node, not your laptop.** Whether `http://<node-ip>:<nodePort>`
> works from your machine depends on your OS: on Linux the minikube node is a container on
> a bridge the host can route to, so it works; on macOS and Windows the Docker daemon runs
> inside its own VM and the node IP is unreachable from the host. That is what
> `minikube service <name> --url` is for — it opens a tunnel — but it **blocks while
> holding that tunnel open**, so it hangs a notebook cell forever. The cell below tries the
> direct route, reports honestly whether it worked on *your* machine, and then uses
> `kubectl port-forward`, which behaves the same everywhere.

In [ ]:
!kubectl expose deployment api-gateway --name api-gateway --type=NodePort --port=8000 --target-port=8000 -n k8s-lab --dry-run=client -o yaml | kubectl apply -f -
!kubectl get svc api-gateway -n k8s-lab

svc = kget("svc", "api-gateway")
node_port = svc["spec"]["ports"][0]["nodePort"]
node_ip = subprocess.run(["minikube", "ip"], capture_output=True, text=True).stdout.strip()
print(f"\nNodePort allocated: {node_port}  (node IP: {node_ip})")

# --- Is that node IP reachable from your laptop? It depends on your OS. ---
# On Linux the minikube node is a container on a bridge the host can route to,
# so http://<minikube ip>:<nodePort> just works. On macOS and Windows, Docker
# itself runs inside a VM, and 192.168.49.2 lives on a network only that VM can
# see -- the connection times out with no useful error. This is why
# `minikube service <svc> --url` exists: it opens a TUNNEL and then BLOCKS
# holding it open, which is exactly what you do NOT want inside a notebook cell.
reachable = http_get(f"http://{node_ip}:{node_port}/health", timeout=4, retries=1)
if reachable and reachable[0] == 200:
    print(f"direct NodePort access works on this machine: {reachable[1]}")
else:
    print(f"direct NodePort access did NOT work here ({reachable[1][:80]}).")
    print("Expected on macOS/Windows with the docker driver -- the node's IP is")
    print("inside Docker's VM. Use `minikube service api-gateway -n k8s-lab`")
    print("(blocks, holding a tunnel open) or `kubectl port-forward`.")

# `kubectl port-forward` is the portable alternative and works on every platform
# and every cluster, so that is what the rest of this notebook uses.
with port_forward("svc/api-gateway", 8000) as base:
    status, body = http_get(f"{base}/health")
print(f"\nvia port-forward: {status} {body}")
assert status == 200 and '"status":"ok"' in body, \
    f"api-gateway did not answer on its Service port: {status} {body[:200]}"
print("✅ NodePort Service created and the app answers behind it")

## Service Discovery With DNS
Kubernetes includes built-in service discovery through DNS. When you create a Service, Kubernetes automatically gives it a DNS name.

The full DNS name pattern looks like this:

`service-name.namespace.svc.cluster.local`

For our user service, that becomes:

`user-service.k8s-lab.svc.cluster.local`

The nice part is that inside the same namespace, you can usually use the short name like `user-service`. Kubernetes expands it for you behind the scenes.

That means your applications can use stable hostnames instead of unstable pod IPs.

In [ ]:
# Same `-i --rm` rule as above.
!kubectl run dns-test -i --rm --restart=Never --image=busybox:1.36 -n k8s-lab -- nslookup user-service.k8s-lab.svc.cluster.local

print()
# The short name resolves too, because every pod's /etc/resolv.conf has a
# `search k8s-lab.svc.cluster.local svc.cluster.local cluster.local` line.
!kubectl run dns-test -i --rm --restart=Never --image=busybox:1.36 -n k8s-lab -- cat /etc/resolv.conf

## Apply the Shared Service Manifests
So far, you created `user-service` and `api-gateway` one at a time so you could see each idea clearly. Now let's apply the shared lab manifest that defines all three Services together:

- `api-gateway` on port 8000 as a `NodePort`
- `user-service` on port 8001 as a `ClusterIP`
- `order-service` on port 8002 as a `ClusterIP`

This manifest lives in `../manifests/service.yaml` relative to this notebook.

In [ ]:
!kubectl apply -f ../manifests/service.yaml
!kubectl get svc -n k8s-lab

print()
# The Service object is only half the picture. THIS is the other half:
!kubectl get endpoints -n k8s-lab

# Each of the three Services should be backed by both of its Deployment's pods.
for name in ("api-gateway", "user-service", "order-service"):
    ips = endpoint_ips(name)
    assert len(ips) == 2, (
        f"Service {name} has {len(ips)} endpoints, expected 2. Either the selector "
        "matches nothing, or the pods are failing readiness."
    )
print("\n✅ all three Services have 2 ready endpoints each")

### The One Debugging Command: `kubectl get endpoints`

A Service does not know anything about Deployments. It is a **label query**. The
Endpoints controller watches for pods whose labels match `spec.selector`, filters that
list down to the ones passing their readiness probe, and writes the survivors' IPs into
an **Endpoints** object with the same name as the Service. `kube-proxy` programs the node
to load-balance to exactly those IPs.

```text
Service.spec.selector      pods with matching labels      pods also passing readiness
   app: user-service   ──▶     3 candidates          ──▶       3 endpoints  ──▶ traffic
```

Break any link in that chain and the Service still exists, still has a ClusterIP, still
resolves in DNS, and `kubectl get svc` still looks perfect. Requests just hang or return
connection refused. Kubernetes will never warn you — a selector that matches nothing is a
completely legal Service.

So: **when a Service does not work, run `kubectl get endpoints <name> -n <ns>` first.**

| `ENDPOINTS` column | Diagnosis |
|---|---|
| A list of `ip:port` | Wiring is fine — the problem is inside your app or the port numbers |
| `<none>` | Either no pod matches the selector, or every matching pod is failing readiness |

Let's cause it on purpose.

In [ ]:
%%writefile ./broken-service.yaml
apiVersion: v1
kind: Service
metadata:
  name: user-service-broken
  namespace: k8s-lab
spec:
  type: ClusterIP
  selector:
    # The pods are labelled `app: user-service`. This says `app: user-svc`.
    # A one-word typo, and this Service will forward traffic to nobody.
    app: user-svc
  ports:
    - name: http
      port: 8001
      targetPort: 8001

In [ ]:
!kubectl apply -f ./broken-service.yaml

print("\n--- looks completely healthy ---")
!kubectl get svc user-service-broken -n k8s-lab

print("\n--- and yet ---")
!kubectl get endpoints user-service-broken -n k8s-lab

print("\n--- describe spells it out ---")
!kubectl describe svc user-service-broken -n k8s-lab | grep -E 'Selector|Endpoints'

print("\n--- what the caller experiences ---")
!kubectl run sel-test -i --rm --restart=Never --image=curlimages/curl:8.7.1 -n k8s-lab --command -- curl -s -m 5 http://user-service-broken:8001/health || echo ">>> no response: the Service has no endpoints (expected)"

# The lesson, asserted: the Service exists, has a ClusterIP, resolves in DNS --
# and has nowhere to send traffic. Nothing about it reports an error.
broken = kget("svc", "user-service-broken")
assert broken["spec"].get("clusterIP") not in (None, "", "None"), \
    "a Service with a nonsense selector is still allocated a ClusterIP"
assert endpoint_ips("user-service-broken") == [], \
    "expected zero endpoints from a selector that matches no pods"
assert len(endpoint_ips("user-service")) == 2, \
    "the correctly-selected Service should be unaffected"
print("\n✅ reproduced: valid Service, real ClusterIP, ZERO endpoints, no error anywhere")

!kubectl delete -f ./broken-service.yaml --ignore-not-found
!rm -f ./broken-service.yaml

`Endpoints: <none>` — that is the whole diagnosis. The second most common cause of the
same symptom is a **port mismatch**: `targetPort` must equal the port the container
actually listens on (`containerPort`), not the Service's own `port`. In our manifests
`port: 8001` and `targetPort: 8001` happen to be equal, which hides the distinction, so
be clear on it:

- `port` — the port the **Service** is reachable on (what callers dial: `user-service:8001`)
- `targetPort` — the port on the **pod** that traffic is forwarded to
- `nodePort` — only for `type: NodePort`; the port opened on every node (30000–32767)

`targetPort` may also be a **port name** rather than a number, which is the more robust
form: name the port `http` in the pod spec and refer to `targetPort: http` in the
Service, and the two can never drift apart. (Naming Service ports matters for another
reason too — a Prometheus `ServiceMonitor` selects a port *by name*, which Notebook 5
relies on.)

## Service-to-Service Calls
Now let's prove that one service can call another service inside the cluster.

The `api-gateway` deployment is already configured with the service URLs:

- `http://user-service:8001`
- `http://order-service:8002`

In the next cell, you will execute a command inside the `api-gateway` container and send a request to `user-service:8001/health`.

The sample images are `python:3.12-slim`, which contains **neither `curl` nor `wget`** — a
very common surprise when you exec into a slim production image. We use Python's
`urllib` instead, which is always present.

In [ ]:
# NOTE: the sample images are built `FROM python:3.12-slim`, which ships with
# neither `curl` nor `wget`. Python is guaranteed to be there, so use it --
# this is a genuinely useful trick for exec'ing into slim application images.
!kubectl exec -n k8s-lab deploy/api-gateway -- python -c "import urllib.request; print(urllib.request.urlopen('http://user-service:8001/health').read().decode())"

print()
# The gateway calling the backend through its own route, i.e. the real path:
!kubectl exec -n k8s-lab deploy/api-gateway -- python -c "import urllib.request; print(urllib.request.urlopen('http://localhost:8000/api/users').read().decode())"

# Assert the full hop actually carried data, not just that the commands ran.
r = subprocess.run(
    ["kubectl", "exec", "-n", NS, "deploy/api-gateway", "--", "python", "-c",
     "import urllib.request;"
     "print(urllib.request.urlopen('http://localhost:8000/api/users').read().decode())"],
    capture_output=True, text=True, timeout=120)
payload = json.loads(r.stdout.strip() or "{}")
assert payload.get("count") == 3 and any(u["name"] == "Alice" for u in payload["users"]), \
    f"api-gateway -> user-service did not return the three sample users: {r.stdout[:300]}"
print(f"\n✅ api-gateway proxied {payload['count']} users out of user-service")

## 🧭 The Four Service Types — and When Each Is Right

| Type | What it creates | Reachable from | Use it when |
|---|---|---|---|
| **ClusterIP** (default) | A virtual IP inside the cluster | Only inside the cluster | Everything internal. This should be ~90% of your Services. |
| **NodePort** | ClusterIP **+** the same port opened on *every* node, 30000–32767 | Anything that can reach a node's IP | Local dev, or as the target of an external load balancer you manage yourself. Rarely the right answer in production: the port range is ugly, and you must know node IPs that change. |
| **LoadBalancer** | NodePort **+** asks the cloud provider for a real external load balancer | The public internet | A cloud cluster (EKS/GKE/AKS), one internet-facing entry point. |
| **ExternalName** | Just a DNS `CNAME`. No proxying, no endpoints. | — | Aliasing an out-of-cluster hostname behind an in-cluster name. |

Note the layering: each type is a superset of the one above it. A `LoadBalancer` Service
still has a ClusterIP and still has a node port.

### LoadBalancer in minikube

`type: LoadBalancer` is implemented by a **cloud controller**. minikube has no cloud
behind it, so a LoadBalancer Service sits at `EXTERNAL-IP: <pending>` forever. That is
not a bug; there is simply nobody to answer the request. `minikube tunnel` (run in a
separate terminal, needs sudo) fakes one by routing a local IP to the Service.

There is also a cost argument: on a cloud, **one LoadBalancer Service = one billed cloud
load balancer**. Twenty microservices with `type: LoadBalancer` means twenty load
balancers and twenty IPs. That is precisely the problem **Ingress** solves — one load
balancer at the edge, routing by host and path to many ClusterIP Services behind it —
which is what we set up next.

### Headless Services: `clusterIP: None`

A normal ClusterIP Service gives you **one** virtual IP and load-balances across pods.
Sometimes you do not want that:

```yaml
spec:
  clusterIP: None      # <- headless
  selector:
    app: redis
```

With `clusterIP: None`, Kubernetes allocates no virtual IP and programs no proxy rules.
Instead, a DNS lookup of the Service name returns **the A records of all ready pods**, and
the client picks one. And when a headless Service is paired with a StatefulSet, each pod
also gets its own stable DNS name: `redis-0.redis.k8s-lab.svc.cluster.local`.

| | ClusterIP | Headless (`clusterIP: None`) |
|---|---|---|
| DNS returns | one virtual IP | one A record per ready pod |
| Load balancing | by kube-proxy, per connection | by the client, however it likes |
| Individual pods addressable | no | yes, with a StatefulSet |
| Right for | stateless HTTP services | databases, quorum members, clients that do their own pooling or sharding |

Notebook 9 uses a headless Service in front of a Redis StatefulSet for exactly this
reason: `redis-0` has to be addressable *as redis-0*.

## Ingress — a Real HTTP Front Door
A `NodePort` is great for learning, but it is a little low-level. In real Kubernetes setups, HTTP traffic usually enters through an **Ingress Controller**.

An Ingress Controller watches `Ingress` resources and turns rules into working HTTP routing. In minikube, the easiest option is the built-in NGINX ingress addon.

Think of the Ingress Controller as the traffic manager at the cluster entrance. It can decide which Service should receive a request based on the path or hostname.

In [ ]:
!minikube addons enable ingress

# The addon creates the controller Deployment, but `addons enable` returns before
# the controller pod is serving and before its admission webhook is reachable.
# Applying an Ingress too early fails with:
#   Internal error occurred: failed calling webhook "validate.nginx.ingress.kubernetes.io"
# So wait for it explicitly. The controller image is ~300 MB, so on a fresh
# cluster this is a real download -- the wait is split across this cell and the
# next so neither blocks longer than a notebook runner allows.
def controller_ready():
    r = subprocess.run(
        ["kubectl", "get", "pods", "-n", "ingress-nginx",
         "-l", "app.kubernetes.io/component=controller", "-o", "json"],
        capture_output=True, text=True)
    if r.returncode != 0:
        return None
    ready = [p for p in json.loads(r.stdout)["items"]
             if any(c["type"] == "Ready" and c["status"] == "True"
                    for c in p["status"].get("conditions", []))]
    print("  ready controller pods:", len(ready))
    return ready or None


first = controller_ready() or wait_until(controller_ready, timeout=150, interval=10,
                                         what="the ingress controller")
print("controller ready" if first else "still pulling -- the next cell keeps waiting")

In [ ]:
# Second half of the wait, then the check.
ready = controller_ready() or wait_until(controller_ready, timeout=170, interval=10,
                                         what="the ingress controller")
assert ready, (
    "the NGINX ingress controller never became Ready. `kubectl get pods -n "
    "ingress-nginx` and `kubectl describe` will say why -- usually a slow image pull."
)

!kubectl get pods -n ingress-nginx
!kubectl get ingressclass

# An Ingress object is inert without a controller AND without an IngressClass to
# select it, so check both exist before writing one.
classes = [c["metadata"]["name"] for c in kget("ingressclass", ns=None)["items"]]
assert "nginx" in classes, f"no `nginx` IngressClass; found {classes}"
print(f"\n✅ ingress controller Ready, IngressClass(es): {classes}")

## Write and Apply the Ingress
We want requests arriving at the cluster to be sent to the `api-gateway` Service. To keep testing simple, this Ingress uses a path rule without a host name, so you can test it directly with the minikube IP.

```text
+-------------+      +-------------------+      +-----------------+
| Browser or  | ---> | NGINX Ingress     | ---> | api-gateway     |
| curl client |      | path-based router |      | Service         |
+-------------+      +-------------------+      +-----------------+
```

The first cell writes the YAML file. The second cell applies it to the cluster.

In [ ]:
%%writefile ./api-gateway-ingress.yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: api-gateway-ingress
  namespace: k8s-lab
spec:
  ingressClassName: nginx
  rules:
    - http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: api-gateway
                port:
                  number: 8000

In [ ]:
!kubectl apply -f ./api-gateway-ingress.yaml
!kubectl get ingress -n k8s-lab
!kubectl describe ingress api-gateway-ingress -n k8s-lab

## Send Traffic Through the Ingress
Now we send a request at the Ingress controller. The controller receives it and forwards
it to `api-gateway`, which handles the `/api/users` route.

If you get JSON back, the full traffic path is working:

`client -> ingress -> service -> pod`

> **Why not `curl http://$(minikube ip)/`?** Because it only works on Linux. With the
> docker driver on macOS or Windows, Docker itself runs inside a VM and the node's IP
> (`192.168.49.2`) is on a network your laptop cannot route to — the request simply times
> out, with no error that hints at the cause. `kubectl port-forward` to the ingress
> controller's own Service exercises exactly the same path and works everywhere, so that
> is what the cell below does.
>
> **If you get a 404 from nginx**: the controller needs a few seconds to notice a new
> Ingress, which is what the retry loop in `http_get` is for. If it persists, check
> `kubectl describe ingress api-gateway-ingress -n k8s-lab` — the `Backends` line should
> name real pod IPs, not `<error: endpoints "api-gateway" not found>`.
>
> Note there is **no `rewrite-target` annotation** on our Ingress, so nginx passes the
> full path through: `/api/users` arrives at api-gateway as `/api/users`. The shared
> `../manifests/ingress.yaml` *does* set `nginx.ingress.kubernetes.io/rewrite-target: /`,
> which rewrites every path to `/` — useful when the backend expects to be mounted at the
> root, and a very confusing 404 generator when it isn't. Know which one you have.

In [ ]:
# The Ingress controller is itself just a Service in the ingress-nginx namespace,
# so port-forwarding to it exercises the complete path
#   client -> ingress controller -> api-gateway Service -> pod
# on every platform. (`curl http://$(minikube ip)/` only works where the host can
# route to the node's IP -- i.e. Linux, not macOS or Windows with the docker
# driver, where it silently times out.)
with port_forward("svc/ingress-nginx-controller", 80, ns="ingress-nginx") as base:
    status, body = http_get(f"{base}/api/users", retries=10)

print(status, body)
payload = json.loads(body)
assert status == 200 and payload.get("count") == 3, \
    f"the request did not survive the trip through the Ingress: {status} {body[:300]}"
print("\n✅ client -> ingress -> service -> pod, end to end")

## 🧹 Clean Up
Use the next cell when you want to remove the Ingress resource created in this notebook. We leave the shared Services in place because later notebooks may still use them.

If you want to keep experimenting, you can skip this cleanup cell for now and run it later.

In [ ]:
!kubectl delete ingress api-gateway-ingress -n k8s-lab --ignore-not-found
!rm -f ./api-gateway-ingress.yaml ./broken-service.yaml

# Note what we do NOT delete: the three Services from ../manifests/service.yaml.
# Notebooks 5 through 10 all rely on them, and re-running this notebook re-applies
# them anyway.
!kubectl get svc -n k8s-lab

## 🎓 What You Learned

Great work. In this notebook, you learned that:

- Pods are temporary, so they are not a safe networking endpoint by themselves
- A **Service** gives your app a stable name and stable virtual IP
- A Service is a **label query**, not a link to a Deployment — and `kubectl get endpoints`
  is the first command to run when one does not work
- **ClusterIP** is the default and should be most of your Services
- **NodePort** opens a port on every node; **LoadBalancer** adds a cloud load balancer and
  stays `<pending>` forever in minikube; **ExternalName** is just a DNS alias
- A **headless** Service (`clusterIP: None`) returns per-pod A records instead of one
  virtual IP — the right choice for StatefulSets
- Kubernetes DNS lets pods discover Services by name, via the `search` domains in
  `/etc/resolv.conf`
- An **Ingress Controller** gives you one HTTP entry point in front of many Services,
  instead of one cloud load balancer per Service
- An **Ingress** resource is inert without a controller running to implement it

If Notebook 03 made sense, you are ready for Notebook 04 where you will package Kubernetes apps with Helm and customize them with Kustomize.